### Created Schema for Silver Layer

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|silver            |
+------------------+



### Transforming Cards Table

In [0]:
cards_bronze_df = spark.table("bronze.cards_bronze")

display(cards_bronze_df.limit(10))

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,true,2,24295.0000,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,true,2,21968.0000,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,true,2,46414.0000,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,false,1,12400.0000,01/2003,2012,No
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,true,1,28.0000,09/2008,2009,No
4537,1746,Visa,Credit,4404898874682993,09/2003,736,true,1,27500.0000,09/2003,2012,No
1278,1746,Visa,Debit,4001482973848631,07/2022,972,true,2,28508.0000,02/2011,2011,No
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,true,2,9022.0000,07/2003,2015,No
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,true,2,54.0000,06/2010,2015,No
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,true,1,99.0000,07/2006,2012,No


In [0]:
cards_bronze_df.printSchema()

root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



#### Null Checks for Cards table

In [0]:
from pyspark.sql import functions as F

cards_bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in cards_bronze_df.columns
]).show()

+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
| id|client_id|card_brand|card_type|card_number|expires|cvv|has_chip|num_cards_issued|credit_limit|acct_open_date|year_pin_last_changed|card_on_dark_web|
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|  0|        0|         0|        0|          0|      0|  0|       0|               0|           0|             0|                    0|               0|
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+



### What Transformation we will do
- convert card_on_dark_web into true/false
- convert acct_open_date from string to date
- create a new column expires_date from the expires string

In [0]:
from pyspark.sql import functions as F

cards_silver_df = (
    cards_bronze_df
    .withColumn(
        "card_on_dark_web",
        F.when(F.col("card_on_dark_web") == "Yes", True).otherwise(False)
    )
    .withColumn(
        "acct_open_date",
        F.to_date(F.col("acct_open_date"), "MM/yyyy")
    )
    .withColumn(
        "expires_date",
        F.to_date(F.concat(F.lit("01/"), F.col("expires")), "dd/MM/yyyy")
    )
)

In [0]:
display(cards_silver_df.limit(10))

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,expires_date
4524,825,Visa,Debit,4344676511950444,12/2022,623,true,2,24295.0000,2002-09-01,2008,false,2022-12-01
2731,825,Visa,Debit,4956965974959986,12/2020,393,true,2,21968.0000,2014-04-01,2014,false,2020-12-01
3701,825,Visa,Debit,4582313478255491,02/2024,719,true,2,46414.0000,2003-07-01,2004,false,2024-02-01
42,825,Visa,Credit,4879494103069057,08/2024,693,false,1,12400.0000,2003-01-01,2012,false,2024-08-01
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,true,1,28.0000,2008-09-01,2009,false,2009-03-01
4537,1746,Visa,Credit,4404898874682993,09/2003,736,true,1,27500.0000,2003-09-01,2012,false,2003-09-01
1278,1746,Visa,Debit,4001482973848631,07/2022,972,true,2,28508.0000,2011-02-01,2011,false,2022-07-01
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,true,2,9022.0000,2003-07-01,2015,false,2022-06-01
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,true,2,54.0000,2010-06-01,2015,false,2020-11-01
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,true,1,99.0000,2006-07-01,2012,false,2023-02-01


In [0]:
cards_silver_df.printSchema()

root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: date (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-- card_on_dark_web: boolean (nullable = false)
 |-- expires_date: date (nullable = true)



### Saving Cards Silver Table

In [0]:
cards_silver_df.write.mode("overwrite").saveAsTable("silver.cards_silver")

In [0]:
spark.sql("SHOW TABLES IN silver").show(truncate=False)

+--------+-------------------+-----------+
|database|tableName          |isTemporary|
+--------+-------------------+-----------+
|silver  |cards_silver       |false      |
|silver  |transactions_silver|false      |
|silver  |users_silver       |false      |
+--------+-------------------+-----------+



In [0]:
display(spark.table("silver.cards_silver").limit(10))

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,expires_date
4524,825,Visa,Debit,4344676511950444,12/2022,623,true,2,24295.0000,2002-09-01,2008,false,2022-12-01
2731,825,Visa,Debit,4956965974959986,12/2020,393,true,2,21968.0000,2014-04-01,2014,false,2020-12-01
3701,825,Visa,Debit,4582313478255491,02/2024,719,true,2,46414.0000,2003-07-01,2004,false,2024-02-01
42,825,Visa,Credit,4879494103069057,08/2024,693,false,1,12400.0000,2003-01-01,2012,false,2024-08-01
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,true,1,28.0000,2008-09-01,2009,false,2009-03-01
4537,1746,Visa,Credit,4404898874682993,09/2003,736,true,1,27500.0000,2003-09-01,2012,false,2003-09-01
1278,1746,Visa,Debit,4001482973848631,07/2022,972,true,2,28508.0000,2011-02-01,2011,false,2022-07-01
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,true,2,9022.0000,2003-07-01,2015,false,2022-06-01
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,true,2,54.0000,2010-06-01,2015,false,2020-11-01
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,true,1,99.0000,2006-07-01,2012,false,2023-02-01


### Transforming Users Table

In [0]:
users_bronze_df = spark.table("bronze.users_bronze")

display(users_bronze_df.limit(10))

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1


In [0]:
users_bronze_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)



#### Null checks for User Table

In [0]:
from pyspark.sql import functions as F

users_bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in users_bronze_df.columns
]).show()

+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|          0|             0|         0|          0|     0|      0|       0|        0|                0|            0|         0|           0|               0|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+



### What needs cleaning in users_silver

#####Main thing:

- income and debt columns are strings with $
- we should convert them to numeric
- everything else already looks fine

In [0]:
from pyspark.sql import functions as F

users_silver_df = (
    users_bronze_df
    .withColumn(
        "per_capita_income_dollar",
        F.regexp_replace("per_capita_income", "[$,]", "").cast("double")
    )
    .withColumn(
        "yearly_income_dollar",
        F.regexp_replace("yearly_income", "[$,]", "").cast("double")
    )
    .withColumn(
        "total_debt_dollar",
        F.regexp_replace("total_debt", "[$,]", "").cast("double")
    )
    .drop("per_capita_income", "yearly_income", "total_debt")
)

In [0]:
display(users_silver_df.limit(10))
users_silver_df.printSchema()

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,credit_score,num_credit_cards,per_capita_income_dollar,yearly_income_dollar,total_debt_dollar
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,787,5,29278.0,59696.0,127613.0
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,701,5,37891.0,77254.0,191349.0
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,698,5,22681.0,33483.0,196.0
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,722,4,163145.0,249925.0,202328.0
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,675,1,53797.0,109687.0,183855.0
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,704,3,20599.0,41997.0,0.0
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,672,3,25258.0,51500.0,102286.0
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,728,1,26790.0,54623.0,114711.0
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,755,5,26273.0,42509.0,2895.0
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,810,1,18730.0,38190.0,81262.0


root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)
 |-- per_capita_income_dollar: double (nullable = true)
 |-- yearly_income_dollar: double (nullable = true)
 |-- total_debt_dollar: double (nullable = true)



### Saving User Details Silver Table

In [0]:
users_silver_df.write.mode("overwrite").saveAsTable("silver.users_silver")

In [0]:
display(spark.table("silver.users_silver").limit(10))

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,credit_score,num_credit_cards,per_capita_income_dollar,yearly_income_dollar,total_debt_dollar
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,787,5,29278.0,59696.0,127613.0
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,701,5,37891.0,77254.0,191349.0
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,698,5,22681.0,33483.0,196.0
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,722,4,163145.0,249925.0,202328.0
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,675,1,53797.0,109687.0,183855.0
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,704,3,20599.0,41997.0,0.0
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,672,3,25258.0,51500.0,102286.0
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,728,1,26790.0,54623.0,114711.0
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,755,5,26273.0,42509.0,2895.0
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,810,1,18730.0,38190.0,81262.0


In [0]:
spark.sql("SHOW TABLES IN silver").show(truncate=False)

+--------+-------------------+-----------+
|database|tableName          |isTemporary|
+--------+-------------------+-----------+
|silver  |cards_silver       |false      |
|silver  |transactions_silver|false      |
|silver  |users_silver       |false      |
+--------+-------------------+-----------+



### Transforming Transcation Table

In [0]:
transactions_bronze_df = spark.table("bronze.transactions_bronze")

display(transactions_bronze_df.limit(10))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01T00:01:00Z,1556,2972,-77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475328,2010-01-01T00:02:00Z,561,4575,14.5700,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,null
7475329,2010-01-01T00:02:00Z,1129,102,80.0000,Swipe Transaction,27092,Vista,CA,92084.0,4829,null
7475331,2010-01-01T00:05:00Z,430,2860,200.0000,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,null
7475332,2010-01-01T00:06:00Z,848,3915,46.4100,Swipe Transaction,13051,Harwood,MD,20776.0,5813,null
7475333,2010-01-01T00:07:00Z,1807,165,4.8100,Swipe Transaction,20519,Bronx,NY,10464.0,5942,null
7475334,2010-01-01T00:09:00Z,1556,2972,77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,5499,null
7475335,2010-01-01T00:14:00Z,1684,2140,26.4600,Online Transaction,39021,ONLINE,null,null,4784,null
7475336,2010-01-01T00:21:00Z,335,5131,261.5800,Online Transaction,50292,ONLINE,null,null,7801,null
7475337,2010-01-01T00:21:00Z,351,1112,10.7400,Swipe Transaction,3864,Flushing,NY,11355.0,5813,null


In [0]:
transactions_bronze_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: short (nullable = true)
 |-- errors: string (nullable = true)



### Null Checks for Transactions table

In [0]:
from pyspark.sql import functions as F

transactions_bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in transactions_bronze_df.columns
]).show()

+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+
| id|date|client_id|card_id|amount|use_chip|merchant_id|merchant_city|merchant_state|    zip|mcc|  errors|
+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+
|  0|   0|        0|      0|     0|       0|          0|            0|       1563700|1652706|  0|13094522|
+---+----+---------+-------+------+--------+-----------+-------------+--------------+-------+---+--------+



### Transaction Table Transformation

For the transaction table, we found null values in `merchant_state`, `zip`, and `errors`. These nulls are expected because online transactions may not have location details, and many transactions do not contain any error. Instead of dropping such rows, we kept the core transaction data and made the table more analysis-ready by renaming `id` to `transaction_id`, creating a `has_error` flag, and replacing null error values with `"No Error"`. We then enriched the table by joining fraud labels from `bronze.fraud_labels_bronze` and merchant category details from `bronze.mcc_codes_bronze`. Finally, we added useful analytical columns such as `transaction_date`, `day_of_week`, `hour_of_day`, `time_of_day`, `week_start`, `year_month`, and `high_value_flag` to support gold-layer fraud analysis.

In [0]:
mcc_bronze_df = spark.table("bronze.mcc_codes_bronze")

display(mcc_bronze_df.limit(10))
mcc_bronze_df.printSchema()

1711,3000,3001,3005,3006,3007,3008,3009,3058,3066,3075,3132,3144,3174,3256,3260,3359,3387,3389,3390,3393,3395,3405,3504,3509,3596,3640,3684,3722,3730,3771,3775,3780,4111,4112,4121,4131,4214,4411,4511,4722,4784,4814,4829,4899,4900,5045,5094,5192,5193,5211,5251,5261,5300,5310,5311,5411,5499,5533,5541,5621,5651,5655,5661,5712,5719,5722,5732,5733,5812,5813,5814,5815,5816,5912,5921,5932,5941,5942,5947,5970,5977,6300,7011,7210,7230,7276,7349,7393,7531,7538,7542,7549,7801,7802,7832,7922,7995,7996,8011,8021,8041,8043,8049,8062,8099,8111,8931,9402
"Heating, Plumbing, Air Conditioning Contractors",Steelworks,Steel Products Manufacturing,Miscellaneous Metal Fabrication,Miscellaneous Fabricated Metal Products,Coated and Laminated Products,Steel Drums and Barrels,Fabricated Structural Metal Products,"Tools, Parts, Supplies Manufacturing",Miscellaneous Metals,"Bolt, Nut, Screw, Rivet Manufacturing",Leather Goods,Floor Covering Stores,Upholstery and Drapery Stores,"Brick, Stone, and Related Materials",Pottery and Ceramics,Non-Ferrous Metal Foundries,"Electroplating, Plating, Polishing Services",Non-Precious Metal Services,Miscellaneous Metalwork,Heat Treating Metal Services,Welding Repair,Ironwork,Gardening Supplies,Industrial Equipment and Supplies,Miscellaneous Machinery and Parts Manufacturing,"Lighting, Fixtures, Electrical Supplies",Semiconductors and Related Devices,Passenger Railways,Ship Chandlers,Railroad Passenger Transport,Railroad Freight,Computer Network Services,Local and Suburban Commuter Transportation,Passenger Railways,Taxicabs and Limousines,Bus Lines,Motor Freight Carriers and Trucking,Cruise Lines,Airlines,Travel Agencies,Tolls and Bridge Fees,Telecommunication Services,Money Transfer,"Cable, Satellite, and Other Pay Television Services","Utilities - Electric, Gas, Water, Sanitary","Computers, Computer Peripheral Equipment",Precious Stones and Metals,"Books, Periodicals, Newspapers","Florists Supplies, Nursery Stock and Flowers",Lumber and Building Materials,Hardware Stores,Lawn and Garden Supply Stores,Wholesale Clubs,Discount Stores,Department Stores,"Grocery Stores, Supermarkets",Miscellaneous Food Stores,Automotive Parts and Accessories Stores,Service Stations,Women's Ready-To-Wear Stores,Family Clothing Stores,"Sports Apparel, Riding Apparel Stores",Shoe Stores,"Furniture, Home Furnishings, and Equipment Stores",Miscellaneous Home Furnishing Stores,Household Appliance Stores,Electronics Stores,Music Stores - Musical Instruments,Eating Places and Restaurants,Drinking Places (Alcoholic Beverages),Fast Food Restaurants,"Digital Goods - Media, Books, Apps",Digital Goods - Games,Drug Stores and Pharmacies,"Package Stores, Beer, Wine, Liquor",Antique Shops,Sporting Goods Stores,Book Stores,"Gift, Card, Novelty Stores","Artist Supply Stores, Craft Shops",Cosmetic Stores,"Insurance Sales, Underwriting","Lodging - Hotels, Motels, Resorts",Laundry Services,Beauty and Barber Shops,Tax Preparation Services,Cleaning and Maintenance Services,"Detective Agencies, Security Services",Automotive Body Repair Shops,Automotive Service Shops,Car Washes,Towing Services,"Athletic Fields, Commercial Sports","Recreational Sports, Clubs",Motion Picture Theaters,Theatrical Producers,"Betting (including Lottery Tickets, Casinos)","Amusement Parks, Carnivals, Circuses","Doctors, Physicians",Dentists and Orthodontists,Chiropractors,"Optometrists, Optical Goods and Eyeglasses",Podiatrists,Hospitals,Medical Services,Legal Services and Attorneys,"Accounting, Auditing, and Bookkeeping Services",Postal Services - Government Only


root
 |-- 1711: string (nullable = true)
 |-- 3000: string (nullable = true)
 |-- 3001: string (nullable = true)
 |-- 3005: string (nullable = true)
 |-- 3006: string (nullable = true)
 |-- 3007: string (nullable = true)
 |-- 3008: string (nullable = true)
 |-- 3009: string (nullable = true)
 |-- 3058: string (nullable = true)
 |-- 3066: string (nullable = true)
 |-- 3075: string (nullable = true)
 |-- 3132: string (nullable = true)
 |-- 3144: string (nullable = true)
 |-- 3174: string (nullable = true)
 |-- 3256: string (nullable = true)
 |-- 3260: string (nullable = true)
 |-- 3359: string (nullable = true)
 |-- 3387: string (nullable = true)
 |-- 3389: string (nullable = true)
 |-- 3390: string (nullable = true)
 |-- 3393: string (nullable = true)
 |-- 3395: string (nullable = true)
 |-- 3405: string (nullable = true)
 |-- 3504: string (nullable = true)
 |-- 3509: string (nullable = true)
 |-- 3596: string (nullable = true)
 |-- 3640: string (nullable = true)
 |-- 3684: string (null

#### What we do next

We will create a cleaned helper DataFrame:

mcc_lookup_df

with 2 columns only:

mcc
merchant_category

Then we can join it to transactions on mcc

In [0]:
from pyspark.sql import functions as F

mcc_bronze_df = spark.table("bronze.mcc_codes_bronze")

mcc_expr = ", ".join([f"'{c}', `{c}`" for c in mcc_bronze_df.columns])

mcc_lookup_df = (
    mcc_bronze_df
    .selectExpr(f"stack({len(mcc_bronze_df.columns)}, {mcc_expr}) as (mcc, merchant_category)")
    .withColumn("mcc", F.col("mcc").cast("int"))
)

In [0]:
display(mcc_lookup_df.limit(20))
mcc_lookup_df.printSchema()

mcc,merchant_category
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


root
 |-- mcc: integer (nullable = true)
 |-- merchant_category: string (nullable = true)



### Inspect fraud_labels_df

In [0]:
fraud_labels_bronze_df = spark.table("bronze.fraud_labels_bronze")

display(fraud_labels_bronze_df.limit(10))
fraud_labels_bronze_df.printSchema()

transaction_id,fraud_label,is_fraud
10649266,No,false
23410063,No,false
9316588,No,false
12478022,No,false
9558530,No,false
12532830,No,false
19526714,No,false
9906964,No,false
13224888,No,false
13749094,No,false


root
 |-- transaction_id: string (nullable = true)
 |-- fraud_label: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)



### One important thing before join

transactions_bronze_df.id is integer
fraud_labels_bronze_df.transaction_id is string

So we need to make the join keys the same type.

In [0]:
from pyspark.sql import functions as F

transactions_base_df = (
    transactions_bronze_df
    .withColumnRenamed("id", "transaction_id")
    .withColumn("transaction_id", F.col("transaction_id").cast("string"))
    .withColumn("has_error", F.when(F.col("errors").isNotNull(), True).otherwise(False))
    .withColumn("errors", F.coalesce(F.col("errors"), F.lit("No Error")))
)

In [0]:
display(transactions_base_df.limit(10))
transactions_base_df.printSchema()

transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,has_error
7475327,2010-01-01T00:01:00Z,1556,2972,-77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,5499,No Error,false
7475328,2010-01-01T00:02:00Z,561,4575,14.5700,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,No Error,false
7475329,2010-01-01T00:02:00Z,1129,102,80.0000,Swipe Transaction,27092,Vista,CA,92084.0,4829,No Error,false
7475331,2010-01-01T00:05:00Z,430,2860,200.0000,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,No Error,false
7475332,2010-01-01T00:06:00Z,848,3915,46.4100,Swipe Transaction,13051,Harwood,MD,20776.0,5813,No Error,false
7475333,2010-01-01T00:07:00Z,1807,165,4.8100,Swipe Transaction,20519,Bronx,NY,10464.0,5942,No Error,false
7475334,2010-01-01T00:09:00Z,1556,2972,77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,5499,No Error,false
7475335,2010-01-01T00:14:00Z,1684,2140,26.4600,Online Transaction,39021,ONLINE,null,null,4784,No Error,false
7475336,2010-01-01T00:21:00Z,335,5131,261.5800,Online Transaction,50292,ONLINE,null,null,7801,No Error,false
7475337,2010-01-01T00:21:00Z,351,1112,10.7400,Swipe Transaction,3864,Flushing,NY,11355.0,5813,No Error,false


root
 |-- transaction_id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: short (nullable = true)
 |-- errors: string (nullable = false)
 |-- has_error: boolean (nullable = false)



### Join fraud labels and MCC lookup

In [0]:
transactions_enriched_df = (
    transactions_base_df.alias("t")
    .join(
        fraud_labels_bronze_df.alias("f"),
        on="transaction_id",
        how="left"
    )
    .join(
        mcc_lookup_df.alias("m"),
        on="mcc",
        how="left"
    )
    .withColumn("is_fraud", F.coalesce(F.col("is_fraud"), F.lit(False)))
    .withColumn("fraud_label", F.coalesce(F.col("fraud_label"), F.lit("No")))
)

In [0]:
display(transactions_enriched_df.limit(10))
transactions_enriched_df.printSchema()

mcc,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,has_error,fraud_label,is_fraud,merchant_category
5499,7475334,2010-01-01T00:09:00Z,1556,2972,77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,No Error,false,No,false,Miscellaneous Food Stores
5942,7475333,2010-01-01T00:07:00Z,1807,165,4.8100,Swipe Transaction,20519,Bronx,NY,10464.0,No Error,false,No,false,Book Stores
4829,7475331,2010-01-01T00:05:00Z,430,2860,200.0000,Swipe Transaction,27092,Crown Point,IN,46307.0,No Error,false,No,false,Money Transfer
4829,7475329,2010-01-01T00:02:00Z,1129,102,80.0000,Swipe Transaction,27092,Vista,CA,92084.0,No Error,false,No,false,Money Transfer
5499,7475327,2010-01-01T00:01:00Z,1556,2972,-77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,No Error,false,No,false,Miscellaneous Food Stores
7801,7475336,2010-01-01T00:21:00Z,335,5131,261.5800,Online Transaction,50292,ONLINE,null,null,No Error,false,No,false,"Athletic Fields, Commercial Sports"
5311,7475328,2010-01-01T00:02:00Z,561,4575,14.5700,Swipe Transaction,67570,Bettendorf,IA,52722.0,No Error,false,No,false,Department Stores
5813,7475332,2010-01-01T00:06:00Z,848,3915,46.4100,Swipe Transaction,13051,Harwood,MD,20776.0,No Error,false,No,false,Drinking Places (Alcoholic Beverages)
5813,7475337,2010-01-01T00:21:00Z,351,1112,10.7400,Swipe Transaction,3864,Flushing,NY,11355.0,No Error,false,No,false,Drinking Places (Alcoholic Beverages)
4784,7475335,2010-01-01T00:14:00Z,1684,2140,26.4600,Online Transaction,39021,ONLINE,null,null,No Error,false,No,false,Tolls and Bridge Fees


root
 |-- mcc: short (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- errors: string (nullable = false)
 |-- has_error: boolean (nullable = false)
 |-- fraud_label: string (nullable = false)
 |-- is_fraud: boolean (nullable = false)
 |-- merchant_category: string (nullable = true)



### Now we only need to add the derived time/business columns that your gold layer will need.

These will help answer questions like:

- fraud by day of week
- fraud rate over time
- fraud by time of day
- weekly fraud users
- monthly spikes
- high-value vs low-value fraud

In [0]:
from pyspark.sql import functions as F

transactions_silver_df = (
    transactions_enriched_df
    .withColumn("transaction_date", F.to_date("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn("hour_of_day", F.hour("date"))
    .withColumn(
        "time_of_day",
        F.when(F.col("hour_of_day").between(5, 11), "Morning")
         .when(F.col("hour_of_day").between(12, 16), "Afternoon")
         .when(F.col("hour_of_day").between(17, 21), "Evening")
         .otherwise("Night")
    )
    .withColumn("week_start", F.date_trunc("week", F.col("date")).cast("date"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn(
        "high_value_flag",
        F.when(F.col("amount") >= 1000, True).otherwise(False)
    )
)

In [0]:
display(transactions_silver_df.limit(10))
transactions_silver_df.printSchema()

mcc,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,has_error,fraud_label,is_fraud,merchant_category,transaction_date,day_of_week,hour_of_day,time_of_day,week_start,year_month,high_value_flag
5499,7475334,2010-01-01T00:09:00Z,1556,2972,77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,No Error,false,No,false,Miscellaneous Food Stores,2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
5942,7475333,2010-01-01T00:07:00Z,1807,165,4.8100,Swipe Transaction,20519,Bronx,NY,10464.0,No Error,false,No,false,Book Stores,2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
4829,7475331,2010-01-01T00:05:00Z,430,2860,200.0000,Swipe Transaction,27092,Crown Point,IN,46307.0,No Error,false,No,false,Money Transfer,2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
4829,7475329,2010-01-01T00:02:00Z,1129,102,80.0000,Swipe Transaction,27092,Vista,CA,92084.0,No Error,false,No,false,Money Transfer,2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
5499,7475327,2010-01-01T00:01:00Z,1556,2972,-77.0000,Swipe Transaction,59935,Beulah,ND,58523.0,No Error,false,No,false,Miscellaneous Food Stores,2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
7801,7475336,2010-01-01T00:21:00Z,335,5131,261.5800,Online Transaction,50292,ONLINE,null,null,No Error,false,No,false,"Athletic Fields, Commercial Sports",2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
5311,7475328,2010-01-01T00:02:00Z,561,4575,14.5700,Swipe Transaction,67570,Bettendorf,IA,52722.0,No Error,false,No,false,Department Stores,2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
5813,7475332,2010-01-01T00:06:00Z,848,3915,46.4100,Swipe Transaction,13051,Harwood,MD,20776.0,No Error,false,No,false,Drinking Places (Alcoholic Beverages),2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
5813,7475337,2010-01-01T00:21:00Z,351,1112,10.7400,Swipe Transaction,3864,Flushing,NY,11355.0,No Error,false,No,false,Drinking Places (Alcoholic Beverages),2010-01-01,Friday,0,Night,2009-12-28,2010-01,false
4784,7475335,2010-01-01T00:14:00Z,1684,2140,26.4600,Online Transaction,39021,ONLINE,null,null,No Error,false,No,false,Tolls and Bridge Fees,2010-01-01,Friday,0,Night,2009-12-28,2010-01,false


root
 |-- mcc: short (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- errors: string (nullable = false)
 |-- has_error: boolean (nullable = false)
 |-- fraud_label: string (nullable = false)
 |-- is_fraud: boolean (nullable = false)
 |-- merchant_category: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- time_of_day: string (nullable = false)
 |-- week_start: date (nullable = true)
 |-- year_month: string (nullable = true)
 |-- high_value_flag: boolean (nullable = false)



### Saving Silver layer Trasaction Table

In [0]:
transactions_silver_df.write.mode("overwrite").saveAsTable("silver.transactions_silver")

In [0]:
display(spark.table("silver.transactions_silver").limit(10))

mcc,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,has_error,fraud_label,is_fraud,merchant_category,transaction_date,day_of_week,hour_of_day,time_of_day,week_start,year_month,high_value_flag
7922,7475468,2010-01-01T03:20:00Z,121,5952,58.4700,Online Transaction,80770,ONLINE,null,null,No Error,false,No,false,Theatrical Producers,2010-01-01,Friday,3,Night,2009-12-28,2010-01,false
5499,7475518,2010-01-01T04:58:00Z,1148,5804,2.6800,Swipe Transaction,59935,Philadelphia,PA,19146.0,No Error,false,No,false,Miscellaneous Food Stores,2010-01-01,Friday,4,Night,2009-12-28,2010-01,false
7349,7475751,2010-01-01T06:52:00Z,1098,4626,46.2500,Swipe Transaction,40616,Battle Creek,IA,51006.0,No Error,false,No,false,Cleaning and Maintenance Services,2010-01-01,Friday,6,Morning,2009-12-28,2010-01,false
4121,7476044,2010-01-01T08:03:00Z,987,4146,34.3100,Online Transaction,18563,ONLINE,null,null,No Error,false,No,false,Taxicabs and Limousines,2010-01-01,Friday,8,Morning,2009-12-28,2010-01,false
5812,7476133,2010-01-01T08:26:00Z,1358,5135,4.1200,Swipe Transaction,65503,Middleton,WI,53562.0,No Error,false,No,false,Eating Places and Restaurants,2010-01-01,Friday,8,Morning,2009-12-28,2010-01,false
3389,7476138,2010-01-01T08:27:00Z,1453,1117,189.8700,Swipe Transaction,16790,Spring Valley,NY,10977.0,No Error,false,No,false,Non-Precious Metal Services,2010-01-01,Friday,8,Morning,2009-12-28,2010-01,false
5812,7476198,2010-01-01T08:42:00Z,1407,5966,6.9400,Swipe Transaction,88646,Bessemer,AL,35023.0,No Error,false,No,false,Eating Places and Restaurants,2010-01-01,Friday,8,Morning,2009-12-28,2010-01,false
5541,7477165,2010-01-01T12:10:00Z,1910,5505,64.0000,Swipe Transaction,61195,Bradenton,FL,34203.0,No Error,false,No,false,Service Stations,2010-01-01,Friday,12,Afternoon,2009-12-28,2010-01,false
5541,7477489,2010-01-01T13:11:00Z,1129,5492,10.3600,Swipe Transaction,41260,Oceanside,CA,92058.0,No Error,false,No,false,Service Stations,2010-01-01,Friday,13,Afternoon,2009-12-28,2010-01,false
5541,7477947,2010-01-01T15:01:00Z,514,1123,10.0000,Swipe Transaction,61195,Indialantic,FL,32903.0,No Error,false,No,false,Service Stations,2010-01-01,Friday,15,Afternoon,2009-12-28,2010-01,false


In [0]:
spark.sql("SHOW TABLES IN silver").show(truncate=False)

+--------+-------------------+-----------+
|database|tableName          |isTemporary|
+--------+-------------------+-----------+
|silver  |cards_silver       |false      |
|silver  |transactions_silver|false      |
|silver  |users_silver       |false      |
+--------+-------------------+-----------+

